# 🌍 Health Financing Dashboard — Data Pipeline

This notebook pulls data from **5 major global health data sources** into a single, clean dataset.

| Source | What it provides | Access |
|--------|-----------------|--------|
| **World Bank** | 55 indicators — health financing, outcomes, macro, demographics | Automatic (API) |
| **WHO GHO** | 40 indicators — UHC, mortality, NCD, HIV/TB/malaria, workforce | Automatic (API) |
| **IMF** | 14 macro indicators — GDP, govt finance, debt, inflation | Automatic (API) |
| **Global Fund** | Grant budgets + programme results by country & disease | Automatic (API) |
| **WHO GHED** | Detailed health expenditure breakdown (20 financing indicators) | **Manual download** (see Step 5) |

**How to use this notebook:**
- Run cells **in order** from top to bottom
- Each section is self-contained and labelled
- Results are saved to `data/processed/` as both `.parquet` and `.csv` files
- You can open the CSV files in Excel to explore the data

---

## Step 1 — Install dependencies

Run this cell **once** to install all required Python packages.
This may take 1–2 minutes on first run.

In [1]:
import subprocess, sys
print('Installing packages...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓ All packages installed successfully.')
else:
    print('⚠ Some packages may have had issues:')
    print(result.stdout[-2000:] if result.stdout else '')
    print(result.stderr[-1000:] if result.stderr else '')

Installing packages...
✓ All packages installed successfully.


## Step 2 — Set up the project

This cell imports all pipeline modules and creates the output directories.

In [2]:
import os, sys, warnings
import pandas as pd
warnings.filterwarnings('ignore')

# Make sure imports resolve correctly regardless of how the notebook was opened
notebook_dir = os.path.abspath('')
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from pipeline.config import PROCESSED_DIR, MANUAL_DIR, RAW_DIR
from pipeline.utils  import load_all_sources, load_data

# Create directories
for d in [PROCESSED_DIR, MANUAL_DIR, RAW_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Setup complete.')
print(f'  Data will be saved to: {PROCESSED_DIR}')

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

---
## Step 3 — World Bank Data

**What you'll get:** 55 indicators across health financing, disease, immunisation, nutrition, WASH, macroeconomics, and demographics.

**Coverage:** 200+ countries | 2000–2023

⏱ *Estimated time: 3–8 minutes depending on internet speed*

In [ ]:
from pipeline.world_bank import fetch_world_bank

df_wb = fetch_world_bank(start_year=2000, end_year=2023, save=True)
print(f'\nPreview:')
df_wb.head(10)

### Quick explore — World Bank
Run this cell to explore what indicators and countries are available.

In [ ]:
if 'df_wb' in dir() and not df_wb.empty:
    print('Indicators available:')
    indicator_summary = df_wb.groupby(['indicator_code', 'indicator_name'])['iso3'].count().reset_index()
    indicator_summary.columns = ['code', 'name', 'country_count']
    display(indicator_summary.sort_values('country_count', ascending=False).head(20))
else:
    print('No World Bank data loaded yet.')

---
## Step 4 — WHO Global Health Observatory

**What you'll get:** 40 indicators covering UHC index, mortality, NCD prevalence, HIV/TB/malaria, immunisation, maternal health, and health workforce.

**Coverage:** 190+ countries | various years

⏱ *Estimated time: 5–10 minutes*

In [ ]:
from pipeline.who_gho import fetch_who_gho

df_gho = fetch_who_gho(start_year=2000, end_year=2023, save=True)
print(f'\nPreview:')
df_gho.head(10)

### Discover more WHO GHO indicators
The WHO GHO has 2,300+ indicators. Run this cell to search for others you might want to add.

In [ ]:
from pipeline.who_gho import list_gho_indicators

# Change the search term to find indicators on any topic
search_term = 'diabetes'
results = list_gho_indicators(search_term)
print(f'Found {len(results)} GHO indicators matching "{search_term}":')
display(results)

---
## Step 5 — IMF Macroeconomic Data

**What you'll get:** 14 macroeconomic indicators — GDP, GDP per capita, government revenue/expenditure, debt, inflation, unemployment.

**Coverage:** 190+ countries | 2000–2023

⏱ *Estimated time: 2–5 minutes*

In [ ]:
from pipeline.imf import fetch_imf

df_imf = fetch_imf(start_year=2000, end_year=2023, save=True)
print(f'\nPreview:')
df_imf.head(10)

### Discover more IMF indicators

In [ ]:
from pipeline.imf import list_imf_indicators

imf_catalogue = list_imf_indicators()
print(f'IMF DataMapper has {len(imf_catalogue)} indicators available.')
display(imf_catalogue.head(30))

---
## Step 6 — Global Fund Data

**What you'll get:** Grant budgets and disbursements by country and disease component (HIV, TB, Malaria, RSSH), plus programme results indicators.

**Coverage:** 100+ countries | grant periods from 2002 onwards

⏱ *Estimated time: 3–8 minutes*

In [ ]:
from pipeline.global_fund import fetch_global_fund

df_gf = fetch_global_fund(save=True)
print(f'\nPreview:')
if df_gf is not None and not df_gf.empty:
    display(df_gf.head(10))
else:
    print('No Global Fund data returned.')

---
## Step 7 — WHO GHED (Manual download required)

**What you'll get:** The most detailed breakdown of health expenditure available anywhere — 20 indicators covering every financing flow (government, OOP, external, insurance, voluntary prepayment, etc.) for 190+ countries.

**Why manual?** The GHED does not have a public API. But the download takes under a minute.

### ⬇️ How to download:
1. Go to **https://apps.who.int/nha/database/**
2. Click the **"Data Explorer"** tab at the top
3. Click **"Download data"** (top right)
4. Select **"All countries"** and **"All years"**
5. Download the Excel file
6. **Save it as:** `data/manual_downloads/GHED_data.xlsx` inside this project folder
7. Then run the cell below

In [ ]:
from pipeline.ghed import process_ghed

df_ghed = process_ghed(start_year=2000, end_year=2023, save=True)
if df_ghed is not None and not df_ghed.empty:
    print(f'\nPreview:')
    display(df_ghed.head(10))

---
## Step 8 — Build the Master Dataset

This cell combines everything into a single master file.

Saved as:
- `data/processed/master.parquet` — for use in Python/dashboard
- `data/processed/master.csv` — open in Excel to explore

In [ ]:
from pipeline.utils import load_all_sources, save_data
from pipeline.config import PROCESSED_DIR

master = load_all_sources(PROCESSED_DIR)
save_data(master, 'master', PROCESSED_DIR)

print(f'\n✓ Master dataset ready')
print(f'  Total rows:        {len(master):,}')
print(f'  Countries:         {master["iso3"].nunique()}')
print(f'  Unique indicators: {master["indicator_code"].nunique()}')
print(f'  Year range:        {master["year"].min()} – {master["year"].max()}')
print(f'\n  Breakdown by source:')
for src in master['source'].unique():
    subset = master[master['source'] == src]
    print(f'    • {src:<25} {len(subset):>8,} rows  |  {subset["indicator_code"].nunique():>3} indicators  |  {subset["iso3"].nunique():>3} countries')
print(f'\nmaster.head()')
master.head(10)

---
## Step 9 — Explore the Data

Use these cells to get a feel for what's in the master dataset before building the dashboard.

In [ ]:
# ── Full indicator catalogue ───────────────────────────────────────────────────
catalogue = (
    master
    .groupby(['indicator_code', 'indicator_name', 'source'])
    .agg(
        countries=('iso3', 'nunique'),
        year_min=('year', 'min'),
        year_max=('year', 'max'),
        total_rows=('value', 'count'),
    )
    .reset_index()
    .sort_values(['source', 'countries'], ascending=[True, False])
)
print(f'Full indicator catalogue ({len(catalogue)} indicators):')
display(catalogue)

In [ ]:
import plotly.express as px

# ── Quick plot: Health expenditure vs GDP per capita ──────────────────────────
# Pivot to get both indicators in the same row for the same (country, year)
pivot = master.pivot_table(
    index=['iso3', 'country_name', 'year'],
    columns='indicator_code',
    values='value',
    aggfunc='first',
).reset_index()

# Pick the most recent year with data for both indicators
health_col = 'SH.XPD.CHEX.GD.ZS'  # Health expenditure % GDP
gdp_col    = 'NY.GDP.PCAP.CD'       # GDP per capita

if health_col in pivot.columns and gdp_col in pivot.columns:
    plot_df = pivot[[health_col, gdp_col, 'iso3', 'country_name', 'year']].dropna()
    plot_df = plot_df[plot_df['year'] == plot_df['year'].max()]

    fig = px.scatter(
        plot_df,
        x=gdp_col,
        y=health_col,
        hover_name='country_name',
        hover_data={'iso3': True, 'year': True},
        log_x=True,
        title=f'Health Expenditure (% GDP) vs GDP per capita — {plot_df["year"].max()}',
        labels={
            gdp_col:    'GDP per capita (USD, log scale)',
            health_col: 'Health expenditure (% of GDP)',
        },
        template='plotly_white',
    )
    fig.show()
else:
    print('Run the World Bank pipeline first to generate this chart.')

In [ ]:
# ── Country deep-dive ─────────────────────────────────────────────────────────
# Change the iso3 code below to explore any country
country_code = 'KEN'  # Kenya

country_data = master[master['iso3'] == country_code].copy()
print(f'Data for {country_code}:')
print(f'  {len(country_data):,} rows  |  {country_data["indicator_code"].nunique()} indicators  |  years {country_data["year"].min()}–{country_data["year"].max()}')
print(f'\nLatest values (most recent year available per indicator):')
latest = (
    country_data
    .sort_values('year')
    .groupby(['indicator_code', 'indicator_name', 'source'])
    .last()
    .reset_index()[['indicator_name', 'year', 'value', 'source']]
    .sort_values('source')
)
display(latest)

In [ ]:
# ── Data coverage heatmap ─────────────────────────────────────────────────────
# Shows which indicators have the best country coverage
coverage = (
    master
    .groupby('indicator_name')['iso3']
    .nunique()
    .reset_index()
    .rename(columns={'iso3': 'countries_with_data'})
    .sort_values('countries_with_data', ascending=False)
)

fig = px.bar(
    coverage.head(40),
    x='countries_with_data',
    y='indicator_name',
    orientation='h',
    title='Top 40 indicators by country coverage',
    labels={'countries_with_data': 'Number of countries', 'indicator_name': ''},
    template='plotly_white',
    height=900,
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

---
## Step 10 — Re-run or refresh individual sources

Once the data is saved, you don't need to re-run everything each time.
Use these cells to reload from the saved files instead.

In [ ]:
# ── Load from saved files (fast — no API calls) ───────────────────────────────
from pipeline.utils  import load_data, load_all_sources
from pipeline.config import PROCESSED_DIR

# Load a specific source
df_wb_cached   = load_data('world_bank', PROCESSED_DIR)
df_gho_cached  = load_data('who_gho',    PROCESSED_DIR)
df_imf_cached  = load_data('imf',        PROCESSED_DIR)
df_gf_cached   = load_data('global_fund',PROCESSED_DIR)

# Or load everything at once
master_cached = load_all_sources(PROCESSED_DIR)

print(f'Loaded master dataset: {len(master_cached):,} rows, {master_cached["indicator_code"].nunique()} indicators')

---
## Adding more indicators

To add indicators from any source:

1. Open `pipeline/config.py`
2. Find the relevant section (e.g. `WORLD_BANK_INDICATORS`, `WHO_GHO_INDICATORS`, `IMF_INDICATORS`)
3. Add a new line: `"INDICATOR_CODE": "Your description"`
4. Re-run the relevant pipeline cell above

Use the discovery cells (Step 4 and Step 5 optional cells) to find indicator codes.

---
## File locations

| File | Description |
|------|-------------|
| `data/processed/master.csv` | All data combined — open in Excel |
| `data/processed/world_bank.csv` | World Bank data only |
| `data/processed/who_gho.csv` | WHO GHO data only |
| `data/processed/imf.csv` | IMF data only |
| `data/processed/global_fund.csv` | Global Fund data only |
| `data/processed/ghed.csv` | WHO GHED data (after manual download) |
| `data/manual_downloads/` | Put manually downloaded files here |